# 04 Weather Pipeline (Synapse-Ready)

**Purpose:** This notebook cleans sensor CSVs from `silver/processing/` and stores the results as Parquet files in `silver/{city}/{domain}/`. It also fetches weather data from the BrightSky API and archives it in `bronze/archive/{city}/weather/`.

**Key difference from local notebooks (01-03):**  
In Azure Synapse, we **cannot** use `pathlib.Path` or the local filesystem. Instead we use **ABFSS paths** (Azure Blob File System) and **`fsspec`** to list and read files. Everything else (pandas, numpy, requests) works the same way.

**Assumptions:**
- CSV files in `silver/processing/` follow the naming pattern `{domain}_{city}.csv` (e.g. `flow_schwartau.csv`).  
- Each city has exactly **4 domain files**: `flow`, `humidity`, `temperature`, `weight`.
- The notebook auto-detects all cities from the filenames — no hardcoding needed!

## 1 — Imports

Same libraries as in notebooks 01/03 — the only new one is **`fsspec`**, which lets us browse files on Azure Data Lake Storage (ADLS) just like we would on a local disk.

In [ ]:
import pandas as pd
import numpy as np
import fsspec          # cloud-aware filesystem (replaces pathlib for ABFSS)
import requests        # HTTP calls for the weather API
import time            # small sleep between retries
from datetime import datetime

## 2 — ABFSS Paths & Storage Credentials

In Synapse, files live on Azure Data Lake Storage. We address them with **ABFSS URIs** instead of local paths like `../data/silver/processing/`.  

The pattern is:  
`abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{folder}`

You can find your account name and key in the Azure Portal:  
**Storage Account → Security + Networking → Access Keys** (either key works).

In [ ]:
# ── Fill in your own values here ──────────────────────────────────────────────
CONTAINER_NAME       = "YOUR_CONTAINER_NAME"
STORAGE_ACCOUNT_NAME = "YOUR_STORAGE_ACCOUNT_NAME"
ACCOUNT_KEY          = "YOUR_ACCOUNT_KEY"

# ── Build the root ABFSS URI (all other paths branch off this) ───────────────
root_dir = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/"

# Source folder where the raw CSVs sit after ingestion
source_path   = root_dir + "silver/processing/"

# Target folders for cleaned outputs
silver_path   = root_dir + "silver/"
bronze_path   = root_dir + "bronze/archive/"

# pandas needs these credentials whenever it reads/writes to ABFSS
storage_options = {
    "account_name": STORAGE_ACCOUNT_NAME,
    "account_key":  ACCOUNT_KEY,
}

## 3 — Discover CSVs & Build Nested Dictionary

On a local machine we used `Path.rglob("*.csv")` and then `path.parent.name` to get the domain/city.  
On ABFSS we cannot do that — instead we use **`fsspec.glob()`** to list files, and plain **string splitting** to extract city and domain from the filename.

Filename convention: `{domain}_{city}.csv`  →  e.g. `flow_schwartau.csv`  
We build a nested dict: **`dfs[city][domain] = DataFrame`**

In [ ]:
# Connect to the storage account with fsspec
fs = fsspec.filesystem("abfss", account_name=STORAGE_ACCOUNT_NAME, account_key=ACCOUNT_KEY)

# List all CSV files in silver/processing/
csv_files = fs.glob(source_path + "*.csv")

# Build a nested dict: dfs[city][domain] = DataFrame
dfs = {}
for filepath in csv_files:
    # filepath looks like: "container@account.../silver/processing/flow_schwartau.csv"
    filename = filepath.split("/")[-1]            # "flow_schwartau.csv"
    name     = filename.replace(".csv", "")        # "flow_schwartau"
    domain   = name.split("_")[0]                  # "flow"
    city     = "_".join(name.split("_")[1:])        # "schwartau" (handles multi-word city names)

    # Create the city sub-dict if it doesn't exist yet
    if city not in dfs:
        dfs[city] = {}

    # Read the CSV into the nested dict
    dfs[city][domain] = pd.read_csv(source_path + filename, storage_options=storage_options)

# Show what we found
for city in dfs:
    print(f"City: {city}  →  domains: {list(dfs[city].keys())}")

## 4 — Outlier Detection Function

Same function as in notebook 01. It uses a **robust z-score** based on the rolling median and MAD (Median Absolute Deviation) instead of mean/std. This makes it resistant to the very outliers we want to catch.

**How it works in plain English:**
1. Compute a rolling median over a window — this is our "expected" value.  
2. Measure how far each point is from that median (absolute deviation).  
3. Compute the rolling median of those deviations — that's the MAD.  
4. Scale MAD by 1.4826 so it's comparable to a standard deviation.  
5. Divide each deviation by the scaled MAD → **robust z-score**.  
6. Any point with |z| > threshold is flagged as an outlier and set to NaN.

In [ ]:
def remove_outliers_zscore(series, window, threshold=3.0):
    """Flag statistical outliers using a rolling MAD-based z-score."""
    # Step 1: rolling median — a robust "centre" that ignores spikes
    rolling_median = series.rolling(window=window, center=True, min_periods=1).median()

    # Step 2: how far is each point from the rolling median?
    abs_deviation = (series - rolling_median).abs()

    # Step 3: rolling median of the deviations → MAD (Median Absolute Deviation)
    mad = abs_deviation.rolling(window=window, center=True, min_periods=1).median()

    # Step 4: scale MAD so it behaves like a standard deviation
    # For normally distributed data:  σ ≈ 1.4826 × MAD
    mad_scaled = 1.4826 * mad

    # Step 5: robust z-score = deviation / scaled MAD
    # Replace 0 with NaN to avoid division-by-zero (happens when the window is constant)
    z_scores = (series - rolling_median) / mad_scaled.replace(0, np.nan)

    # Step 6: mark anything with |z| above the threshold as outlier → NaN
    is_outlier = z_scores.abs() > threshold
    cleaned = series.copy()
    cleaned[is_outlier] = np.nan

    return cleaned

## 5 — Cleaning Functions (one per domain)

Below we define **four separate functions** — one for each sensor domain. Each takes a raw DataFrame and returns a cleaned one. This keeps the domain-specific logic isolated and easy to read, while letting us loop through all cities in a single cell later.

All four follow the same general recipe:  
**parse timestamp → localize to UTC → sort → deduplicate → remove outliers → interpolate → keep only clean columns**

But each has its own quirks — read the inline comments!

In [ ]:
def clean_flow(df):
    """Clean flow data: split by sign, remove outliers, merge, interpolate."""
    flow_raw = df.copy()

    # Flow timestamps are already in UTC format — just parse them
    flow_raw["timestamp"] = pd.to_datetime(flow_raw["timestamp"], errors="coerce", utc=True)

    # Split by sign: negative → flow_in, positive → flow_out (zeros excluded)
    flow_in  = flow_raw.loc[flow_raw["flow"] < 0].copy().reset_index(drop=True)
    flow_out = flow_raw.loc[flow_raw["flow"] > 0].copy().reset_index(drop=True)

    # Remove outliers from each direction independently (window=60 → 1 hour at 1-min sampling)
    flow_in["flow_clean"]  = remove_outliers_zscore(flow_in["flow"],  window=60, threshold=3.0)
    flow_out["flow_clean"] = remove_outliers_zscore(flow_out["flow"], window=60, threshold=3.0)

    # Merge both directions back together on timestamp
    flow_merged = (
        flow_in[["timestamp", "flow_clean"]]
        .rename(columns={"flow_clean": "flow_in"})
        .merge(
            flow_out[["timestamp", "flow_clean"]].rename(columns={"flow_clean": "flow_out"}),
            on="timestamp",
            how="outer",
        )
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    # Drop exact duplicate timestamps (keep first occurrence)
    flow_merged = flow_merged.drop_duplicates(subset="timestamp", keep="first")

    # Set timestamp as index and enforce regular 1-minute frequency
    flow_merged = flow_merged.set_index("timestamp").asfreq("min")

    # Interpolate small gaps (up to 60 consecutive minutes)
    for col in ["flow_in", "flow_out"]:
        flow_merged[col] = flow_merged[col].interpolate(method="time", limit=60, limit_area="inside")

    return flow_merged.reset_index()

In [ ]:
def clean_humidity(df):
    """Clean humidity: localize tz, remove outliers, clamp 0-100%, interpolate."""
    humidity = df.copy()

    # Convert timestamp and localize Berlin → UTC
    # ambiguous=True: during DST overlap, assume the earlier (summer) time
    humidity["timestamp"] = pd.to_datetime(humidity["timestamp"], errors="coerce")
    humidity["timestamp"] = (
        humidity["timestamp"]
        .dt.tz_localize("Europe/Berlin", ambiguous=True)
        .dt.tz_convert("UTC")
    )

    # Sort and remove duplicate timestamps
    humidity = humidity.sort_values("timestamp").drop_duplicates(subset="timestamp", keep="first")

    # Outlier removal (window=5 → ~2.5 days at 12 h intervals)
    humidity["humidity_clean"] = remove_outliers_zscore(humidity["humidity"], window=5, threshold=3.0)

    # Clamp to physical range: humidity must be between 0 and 100 %
    impossible = (humidity["humidity_clean"] < 0) | (humidity["humidity_clean"] > 100)
    humidity.loc[impossible, "humidity_clean"] = np.nan

    # Interpolate small gaps (up to 2 readings = 1 day)
    humidity = humidity.set_index("timestamp")
    humidity["humidity_clean"] = humidity["humidity_clean"].interpolate(method="time", limit=2, limit_area="inside")
    humidity = humidity.reset_index()

    # Keep only cleaned column, rename back to "humidity"
    return humidity[["timestamp", "humidity_clean"]].rename(columns={"humidity_clean": "humidity"})

In [ ]:
def clean_temperature(df):
    """Clean temperature: localize tz (infer DST), remove outliers, interpolate."""
    temperature = df.copy()

    # Convert timestamp and localize Berlin → UTC
    # ambiguous="infer": 5-min sampling gives pandas enough data points to figure out DST
    temperature["timestamp"] = pd.to_datetime(temperature["timestamp"], errors="coerce")
    temperature["timestamp"] = (
        temperature["timestamp"]
        .dt.tz_localize("Europe/Berlin", ambiguous="infer")
        .dt.tz_convert("UTC")
    )
    temperature = temperature.sort_values("timestamp").drop_duplicates(subset="timestamp", keep="first")

    # Outlier removal (window=60 → 5 hours at 5-min intervals)
    temperature["temperature_clean"] = remove_outliers_zscore(temperature["temperature"], window=60, threshold=3.0)

    # Interpolate gaps up to 60 readings (= 5 hours)
    temperature = temperature.set_index("timestamp")
    temperature["temperature_clean"] = temperature["temperature_clean"].interpolate(method="time", limit=60, limit_area="inside")
    temperature = temperature.reset_index()

    # Keep only cleaned column
    return temperature[["timestamp", "temperature_clean"]].rename(columns={"temperature_clean": "temperature"})

In [ ]:
def clean_weight(df):
    """Clean weight: localize tz, remove outliers, interpolate."""
    weight = df.copy()

    # Convert timestamp and localize Berlin → UTC
    # ambiguous=True: same approach as humidity (12 h sampling, can't infer DST)
    weight["timestamp"] = pd.to_datetime(weight["timestamp"], errors="coerce")
    weight["timestamp"] = (
        weight["timestamp"]
        .dt.tz_localize("Europe/Berlin", ambiguous=True)
        .dt.tz_convert("UTC")
    )
    weight = weight.sort_values("timestamp").drop_duplicates(subset="timestamp", keep="first")

    # Outlier removal (window=14 → 1 week at 12 h intervals)
    weight["weight_clean"] = remove_outliers_zscore(weight["weight"], window=14, threshold=3.0)

    # Interpolate gaps up to 8 readings (= 4 days)
    weight = weight.set_index("timestamp")
    weight["weight_clean"] = weight["weight_clean"].interpolate(method="time", limit=8, limit_area="inside")
    weight = weight.reset_index()

    # Keep only cleaned column
    return weight[["timestamp", "weight_clean"]].rename(columns={"weight_clean": "weight"})

## 6 — Clean All Cities & Save to Silver

Now we simply loop through every city and call the four functions defined above. The cleaned DataFrames are stored back in `dfs[city][domain]` and written as Parquet to:  
`silver/{city}/{domain}/{domain}_{timestamp}.parquet`

In [ ]:
# Timestamp for the output filenames (records when this pipeline ran)
now_str = datetime.utcnow().strftime("%Y-%m-%d_%Hh%Mm%Ss")

# The four sensor domains and their matching cleaning functions
DOMAINS = ["flow", "humidity", "temperature", "weight"]
CLEAN_FN = {
    "flow":        clean_flow,
    "humidity":    clean_humidity,
    "temperature": clean_temperature,
    "weight":      clean_weight,
}

for city in dfs:
    print(f"═══ Processing city: {city} ═══")

    for domain in DOMAINS:
        # Clean the raw data using the domain-specific function
        dfs[city][domain] = CLEAN_FN[domain](dfs[city][domain])

        # Build the output path and save as Parquet
        out_path = f"{silver_path}{city}/{domain}/{domain}_{now_str}.parquet"
        dfs[city][domain].to_parquet(out_path, index=False, storage_options=storage_options)

        print(f"  {domain:>12}  shape: {str(dfs[city][domain].shape):<16}  → {out_path}")

## 7 — Weather API: Setup

To fetch weather data we need the **GPS coordinates** of each city and the **date range** covered by our sensor data. We store coordinates in a lookup dictionary — add new cities here when you expand the project.

In [ ]:
# ── City coordinates (add new cities here!) ──────────────────────────────────
# Each entry maps a city name (as it appears in the filenames) to (lat, lon).
CITY_COORDINATES = {
    "schwartau": (53.919444, 10.6975),
    # "berlin":  (52.5200, 13.4050),   # ← example: uncomment when you add Berlin
}

# ── Determine the date range per city from the cleaned sensor data ───────────
# We look at the min/max timestamps across all four domains for each city.
for city in dfs:
    all_mins = []
    all_maxs = []
    for domain in DOMAINS:
        ts = dfs[city][domain]["timestamp"]
        all_mins.append(ts.min())
        all_maxs.append(ts.max())

    # Store the overall date range alongside the DataFrames
    dfs[city]["start_dt"] = min(all_mins)
    dfs[city]["end_dt"]   = max(all_maxs)

    print(f"{city}: {dfs[city]['start_dt']}  →  {dfs[city]['end_dt']}")

## 8 — Weather API: Fetch Function

We use the free **BrightSky API** (a wrapper around DWD open data). The function below fetches hourly weather for a given location and date range.

**Why retries?** In a Synapse pipeline, a transient network hiccup would crash the entire notebook and you'd have to re-run everything from scratch. Adding a few retries with a small back-off delay makes the pipeline much more robust — at almost no extra complexity.

In [ ]:
BRIGHTSKY_API    = "https://api.brightsky.dev/weather"
TIMEOUT_SECONDS  = 30
RETRIES          = 3
MAX_DISTANCE     = 50_000   # metres — search radius for nearest DWD station

WEATHER_COLUMNS = [
    "timestamp",
    "precipitation",
    "pressure_msl",
    "sunshine",
    "temperature",
    "wind_direction",
    "wind_speed",
    "cloud_cover",
    "dew_point",
    "relative_humidity",
    "wind_gust_direction",
    "wind_gust_speed",
    "solar",
]


def fetch_weather(lat, lon, start, end):
    """Fetch hourly weather JSON from BrightSky for a coordinate + date range."""
    params = {
        "lat": lat,
        "lon": lon,
        "date": start,
        "last_date": end,
        "tz": "UTC",
        "units": "dwd",
        "max_dist": MAX_DISTANCE,
    }

    # Retry loop — small back-off between attempts so transient errors don't kill the pipeline
    for attempt in range(1, RETRIES + 1):
        response = requests.get(BRIGHTSKY_API, params=params, timeout=TIMEOUT_SECONDS)
        if response.status_code == 200:
            return response.json()
        # Wait a little longer on each retry (0.3s, 0.6s, 0.9s)
        time.sleep(0.3 * attempt)

    raise RuntimeError(f"BrightSky API failed after {RETRIES} attempts (status {response.status_code})")

## 9 — Fetch Weather & Save to Bronze Archive

For each city we:
1. Call the BrightSky API with the city's coordinates and date range.
2. Save the raw JSON response as Parquet into `bronze/archive/{city}/weather/` — this is our immutable archive layer.

In [ ]:
for city in dfs:
    lat, lon = CITY_COORDINATES[city]
    start_dt = dfs[city]["start_dt"]
    end_dt   = dfs[city]["end_dt"]

    # Format dates as strings for the API
    start_str = start_dt.strftime("%Y-%m-%d")
    end_str   = end_dt.strftime("%Y-%m-%d")

    print(f"── Fetching weather for {city} ({start_str} → {end_str}) ...")
    raw_json = fetch_weather(lat, lon, start_str, end_str)

    # Convert JSON arrays to DataFrames
    raw_weather = pd.json_normalize(raw_json["weather"])
    raw_sources = pd.json_normalize(raw_json["sources"])

    # Build bronze archive paths
    bronze_city_path = f"{bronze_path}{city}/weather/"
    weather_file = f"{bronze_city_path}weather_raw_{start_str}_{end_str}.parquet"
    sources_file = f"{bronze_city_path}sources_raw_{start_str}_{end_str}.parquet"

    # Save raw data to bronze (immutable archive)
    raw_weather.to_parquet(weather_file, index=False, storage_options=storage_options)
    raw_sources.to_parquet(sources_file, index=False, storage_options=storage_options)

    # Store references for the next step
    dfs[city]["bronze_weather"] = raw_weather
    dfs[city]["bronze_sources"] = raw_sources

    print(f"   Bronze saved → {weather_file}")
    print(f"   Raw weather shape: {raw_weather.shape}, sources shape: {raw_sources.shape}")

## 10 — Process Weather Data (Bronze → Silver)

The raw BrightSky response contains station metadata (primary + fallback stations). We:
1. Resolve which weather station was actually used (primary vs. fallback).
2. Attach station name and distance.
3. Select only the columns we need.
4. Clean the timestamp and save as Parquet to `silver/{city}/weather/`.

In [ ]:
for city in dfs:
    print(f"── Processing weather for: {city}")
    bronze_weather = dfs[city]["bronze_weather"].copy()
    bronze_sources = dfs[city]["bronze_sources"].copy()

    # Start with the primary source station
    bronze_weather["station_id"] = bronze_weather["source_id"]

    # Check if a fallback station was used for temperature readings
    if "fallback_source_ids.temperature" in bronze_weather.columns:
        bronze_weather["fallback_station_id"] = bronze_weather["fallback_source_ids.temperature"]
    else:
        bronze_weather["fallback_station_id"] = pd.NA

    # Build lookup tables from the sources metadata
    source_lookup = bronze_sources.drop_duplicates(subset=["id"]).set_index("id")
    distance_lookup     = source_lookup["distance"]
    station_name_lookup = source_lookup["station_name"]

    # Map distance and station name from the primary station
    bronze_weather["distance"]     = bronze_weather["source_id"].map(distance_lookup)
    bronze_weather["station_name"] = bronze_weather["source_id"].map(station_name_lookup)

    # Where a fallback station exists, override with its metadata
    fallback_mask = bronze_weather["fallback_station_id"].notna()
    bronze_weather.loc[fallback_mask, "station_id"]    = bronze_weather.loc[fallback_mask, "fallback_station_id"]
    bronze_weather.loc[fallback_mask, "distance"]      = bronze_weather.loc[fallback_mask, "fallback_station_id"].map(distance_lookup)
    bronze_weather.loc[fallback_mask, "station_name"]  = bronze_weather.loc[fallback_mask, "fallback_station_id"].map(station_name_lookup)

    # Select final columns: core weather variables + station metadata
    final_columns = WEATHER_COLUMNS + [
        col for col in ["station_id", "distance", "station_name"]
        if col in bronze_weather.columns
    ]
    weather_df = bronze_weather[final_columns].copy()

    # Clean the timestamp column
    weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"], errors="coerce", utc=True)
    weather_df = (
        weather_df
        .dropna(subset=["timestamp"])
        .sort_values("timestamp")
        .drop_duplicates(subset=["timestamp"], keep="last")
        .reset_index(drop=True)
    )

    # Save to silver/{city}/weather/
    start_str = dfs[city]["start_dt"].strftime("%Y-%m-%d")
    end_str   = dfs[city]["end_dt"].strftime("%Y-%m-%d")
    weather_out = f"{silver_path}{city}/weather/weather_historical_hourly_{start_str}_{end_str}.parquet"
    weather_df.to_parquet(weather_out, index=False, storage_options=storage_options)

    # Also store in our dict for downstream use
    dfs[city]["weather"] = weather_df

    print(f"   → weather shape: {weather_df.shape}")
    print(f"   Silver saved → {weather_out}")

## Done!

The pipeline has completed. Here is a summary of what was written:

| Layer | Path | Content |
|-------|------|---------|
| **Silver** | `silver/{city}/flow/flow_{timestamp}.parquet` | Cleaned flow (in/out), 1-min frequency |
| **Silver** | `silver/{city}/humidity/humidity_{timestamp}.parquet` | Cleaned humidity |
| **Silver** | `silver/{city}/temperature/temperature_{timestamp}.parquet` | Cleaned temperature |
| **Silver** | `silver/{city}/weight/weight_{timestamp}.parquet` | Cleaned weight |
| **Silver** | `silver/{city}/weather/weather_historical_hourly_{start}_{end}.parquet` | Processed hourly weather |
| **Bronze** | `bronze/archive/{city}/weather/weather_raw_{start}_{end}.parquet` | Raw API response (archive) |
| **Bronze** | `bronze/archive/{city}/weather/sources_raw_{start}_{end}.parquet` | Station metadata (archive) |

To add a new city, just place its 4 CSV files in `silver/processing/` (e.g. `flow_berlin.csv`, ...) and add its coordinates to the `CITY_COORDINATES` dictionary in cell 10.